In [1]:
# In cmd.exe
# .venv\Scripts\activate.bat

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


import torch
import torch.nn.functional as F
from torch import nn
from torch import Tensor
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

In [3]:
# https://www.kaggle.com/datasets/infamouscoder/dataset-netflix-shows
# https://www.kaggle.com/code/samad0015/eda-on-netflix-shows
path = "data/netflix_titles.csv"
df = pd.read_csv(path)

df.shape

(8807, 12)

In [4]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [5]:
states = []

for line in df['country'].unique():
    if pd.isna(line):
        states.append("undefined")
        continue 
    for state in line.replace(", ", ",").split(","):
        states.append(state)


np.unique(np.array(states))

array(['', 'Afghanistan', 'Albania', 'Algeria', 'Angola', 'Argentina',
       'Armenia', 'Australia', 'Austria', 'Azerbaijan', 'Bahamas',
       'Bangladesh', 'Belarus', 'Belgium', 'Bermuda', 'Botswana',
       'Brazil', 'Bulgaria', 'Burkina Faso', 'Cambodia', 'Cameroon',
       'Canada', 'Cayman Islands', 'Chile', 'China', 'Colombia',
       'Croatia', 'Cuba', 'Cyprus', 'Czech Republic', 'Denmark',
       'Dominican Republic', 'East Germany', 'Ecuador', 'Egypt',
       'Ethiopia', 'Finland', 'France', 'Georgia', 'Germany', 'Ghana',
       'Greece', 'Guatemala', 'Hong Kong', 'Hungary', 'Iceland', 'India',
       'Indonesia', 'Iran', 'Iraq', 'Ireland', 'Israel', 'Italy',
       'Jamaica', 'Japan', 'Jordan', 'Kazakhstan', 'Kenya', 'Kuwait',
       'Latvia', 'Lebanon', 'Liechtenstein', 'Lithuania', 'Luxembourg',
       'Malawi', 'Malaysia', 'Malta', 'Mauritius', 'Mexico', 'Mongolia',
       'Montenegro', 'Morocco', 'Mozambique', 'Namibia', 'Nepal',
       'Netherlands', 'New Zealand', 'Ni

Try GPT

In [6]:
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
import tqdm

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Set pad token to eos_token to avoid warning
tokenizer.pad_token = tokenizer.eos_token

prompt = "Brad Pitt"

inputs = tokenizer(prompt, return_tensors="pt", padding=True, return_attention_mask=True)

with torch.no_grad():
    outputs = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        pad_token_id=tokenizer.eos_token_id,
        max_new_tokens=50,
        do_sample=True,  # optional: adds randomness
        top_k=50,
        top_p=0.95
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)



c:\Users\david\Università e appunti\Università\IV anno, I semestre\Machine Learning Operations\Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Brad Pitt

A couple of months ago I wrote that one of my favorite parts of my career was working on a show that I wanted to write. We had a good relationship in my school, so I wanted to try and make the show as personal and


Without context

In [7]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Tokenize input
texts = ["Brad Pitt", "Edward Norton", "Helena Bonham Carter"]
inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)

# Pass token IDs into the embedding layer directly
token_ids = inputs["input_ids"]  # shape: (batch_size, seq_len)
token_embeddings = model.get_input_embeddings()(token_ids)  # shape: (batch_size, seq_len, hidden_size)

# Optional: Mean pool across tokens
sentence_embeddings = token_embeddings.mean(dim=1)

print("Embedding shape:", sentence_embeddings.shape)  # e.g., torch.Size([3, 768])


Embedding shape: torch.Size([3, 768])


With context

In [8]:

# Load a BERT model (for embeddings)
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# List of strings (e.g. actor names)
texts = ["Brad Pitt", "Edward Norton", "Helena Bonham Carter"]

# Tokenize
inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)

# Disable gradient tracking
with torch.no_grad():
    outputs = model(**inputs)
    token_embeddings = outputs.last_hidden_state  # shape: (batch_size, seq_len, hidden_dim)

# Mean pool over tokens (dim=1)
sentence_embeddings = token_embeddings.mean(dim=1)

print("Embedding shape:", sentence_embeddings.shape)  # torch.Size([3, 768])

Embedding shape: torch.Size([3, 768])


In [9]:
for line in df['country']:
    print(line)

United States
South Africa
nan
nan
India
nan
nan
United States, Ghana, Burkina Faso, United Kingdom, Germany, Ethiopia
United Kingdom
United States
nan
nan
Germany, Czech Republic
nan
nan
United States
nan
Mexico
nan
nan
nan
Turkey
nan
nan
India
Australia
nan
United States
United States
United States, India, France
nan
nan
United Kingdom
nan
nan
nan
nan
Finland
China, Canada, United States
India
United States
United States
United States
United States
United States
nan
South Africa, United States, Japan
nan
United States
Nigeria
India
Japan
Japan
Japan
Japan
United States
Japan
Japan
Japan
Japan
Japan
Japan
Japan
Japan
nan
United Kingdom
India
United States
nan
India
nan
nan
United Kingdom
Nigeria
nan
nan
Japan
nan
nan
nan
nan
United States
United States
nan
Nigeria
nan
nan
nan
nan
nan
Spain, United States
France
Belgium
nan
United Kingdom, United States
United States, United Kingdom
United States
United States
United Kingdom
France, United States
nan
United States
nan
nan
South Korea
I